# Notebook 13 — Warum *dieses* Modell? (Modellauswahl, visuell)

Dieses Notebook belegt **visuell**, welche Kombination das beste Modell für die
Speise-Klassifikation (Apfel / Kaugummi / Skyr) liefert. Variiert werden systematisch:

- **Feature-Set:** PLUS-52 (alle) vs. **SELECTED** (group-aware Feature Selection)
- **Movement Exclusion:** aus vs. an
- **Bewertung:** 80/20-Fenster-Split · LOSO (Session-weise, ehrlich) · Per-Mahlzeit-Votum

Modell durchgehend: **SVM (RBF, C=10)**. Drei Grafiken: Vergleichsmatrix · Feature-Importance ·
Confusion des besten Modells.

---
## 1. Setup & Features

In [ ]:
import zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.signal import welch, find_peaks
from scipy.stats import kurtosis, skew
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit, train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix

warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

FS=50.0; TRIM=2; WIN=10.0; STEP=10.0; MINTAIL=8.0; K_ME=5
RAW=['Apfel','Kaugummi','Skyr','Skyr_Zimt']
def fine(c): return 'Skyr' if c=='Skyr_Zimt' else c
SPEC=['Apfel','Kaugummi','Skyr']
NEW={'mag_kurtosis','mag_skew','crest_factor','jerk_mean','jerk_std','spectral_entropy',
     'spectral_centroid','spectral_bandwidth','band_power_slow','band_power_mid',
     'band_power_fast','high_freq_power','ac_peak_height','chew_freq_ac','chews_per_sec','inter_chew_cv'}
DATA=Path('../data/raw'); SKIP={'Metadata.csv','Annotation.csv'}

def prep(df):
    df=df.copy(); t=df['seconds_elapsed']
    df=df[(t>=t.iloc[0]+TRIM)&(t<=t.iloc[-1]-TRIM)].reset_index(drop=True)
    df['lin_x']=df['accelerationX']; df['lin_y']=df['accelerationY']; df['lin_z']=df['accelerationZ']
    df['magnitude']=np.sqrt(df['lin_x']**2+df['lin_y']**2+df['lin_z']**2); return df
def me_mask(df):
    thr=max(0.02,K_ME*df['magnitude'].median())
    return df[df['magnitude'].rolling(50,center=True,min_periods=1).max()<=thr].reset_index(drop=True)
def wins(df):
    t=df['seconds_elapsed'].values; ts,te=t[0],t[-1]; out=[]
    while ts+MINTAIL<=te:
        w=df[(t>=ts)&(t<ts+WIN)].reset_index(drop=True)
        if len(w)>1 and (w['seconds_elapsed'].iloc[-1]-w['seconds_elapsed'].iloc[0])>=MINTAIL: out.append(w)
        ts+=STEP
    return out
def feat(df):
    f={}
    for c in ['lin_x','lin_y','lin_z','magnitude']:
        f[c+'_mean']=df[c].mean(); f[c+'_std']=df[c].std(); f[c+'_max']=df[c].abs().max()
    f['stillness_ratio']=(df['magnitude']<0.02).mean(); f['movement_events']=int((df['magnitude']>df['magnitude'].quantile(0.75)).sum())
    for c in ['rotationRateX','rotationRateY','rotationRateZ']:
        f[c+'_mean']=df[c].mean(); f[c+'_std']=df[c].std(); f[c+'_max']=df[c].abs().max()
    for c in ['pitch','roll','yaw']:
        f[c+'_mean']=df[c].mean(); f[c+'_std']=df[c].std(); f[c+'_range']=df[c].max()-df[c].min()
    npg=min(256,len(df)//2); fr,psd=welch(df['magnitude'].values,fs=FS,nperseg=npg)
    ch=(fr>=0.5)&(fr<=4.0); cf,cp=fr[ch],psd[ch]
    f['total_power']=float(psd.sum()); f['chew_band_power']=float(cp.sum())
    f['rhythmicity']=f['chew_band_power']/f['total_power'] if f['total_power']>0 else 0.0
    f['dominant_chew_freq']=float(cf[np.argmax(cp)]) if len(cp)>0 else 0.0
    mag=df['magnitude'].values; t=df['seconds_elapsed'].values; dur=(t[-1]-t[0]) if len(t)>1 else 0.0
    rms=np.sqrt(np.mean(mag**2)) if len(mag) else 0.0
    f['mag_kurtosis']=float(kurtosis(mag)) if len(mag)>3 else 0.0; f['mag_skew']=float(skew(mag)) if len(mag)>2 else 0.0
    f['crest_factor']=float(np.max(np.abs(mag))/rms) if rms>1e-9 else 0.0
    if len(mag)>2:
        jk=np.diff(mag)*FS; f['jerk_mean']=float(np.mean(np.abs(jk))); f['jerk_std']=float(np.std(jk))
    else: f['jerk_mean']=f['jerk_std']=0.0
    for k in ['spectral_entropy','spectral_centroid','spectral_bandwidth','band_power_slow','band_power_mid','band_power_fast','high_freq_power']: f[k]=0.0
    if npg>=2 and psd.sum()>0:
        tot=psd.sum(); p=psd/tot; pp=p[p>0]
        f['spectral_entropy']=float(-np.sum(pp*np.log2(pp))/np.log2(len(p))); f['spectral_centroid']=float(np.sum(fr*p))
        f['spectral_bandwidth']=float(np.sqrt(np.sum((fr-f['spectral_centroid'])**2*p)))
        bd=lambda lo,hi: float(psd[(fr>=lo)&(fr<hi)].sum()/tot)
        f['band_power_slow']=bd(0.5,1.5); f['band_power_mid']=bd(1.5,2.5); f['band_power_fast']=bd(2.5,4.0); f['high_freq_power']=bd(4.0,15.0)
    f['ac_peak_height']=0.0; f['chew_freq_ac']=0.0
    sig=mag-mag.mean() if len(mag) else mag
    if len(sig)>int(FS):
        ac=np.correlate(sig,sig,mode='full')[len(sig)-1:]; ac=ac/(ac[0]+1e-12); lo,hi=int(FS/4.0),min(int(FS/0.5),len(ac)-1)
        if hi>lo: pk=np.argmax(ac[lo:hi])+lo; f['ac_peak_height']=float(ac[pk]); f['chew_freq_ac']=float(FS/pk)
    f['chews_per_sec']=0.0; f['inter_chew_cv']=0.0
    if len(mag)>5 and mag.std()>0:
        pks,_=find_peaks(mag,distance=max(1,int(FS*0.2)),prominence=mag.std())
        if dur>0: f['chews_per_sec']=float(len(pks)/dur)
        if len(pks)>1:
            iv=np.diff(t[pks])
            if iv.mean()>0: f['inter_chew_cv']=float(iv.std()/iv.mean())
    return f
print('Setup OK')

---
## 2. Datensätze bauen (ME aus + ME an) & Feature Selection

Für jede ME-Einstellung werden die Features berechnet und per **group-aware Permutation
Importance** selektiert (Features mit negativem Beitrag zur Cross-Session-Vorhersage fliegen raus).

In [ ]:
sessions=[]
for zf in sorted(DATA.glob('*.zip')):
    nm=zf.name
    for cls in sorted(RAW,key=len,reverse=True):
        if nm.startswith(cls+'_') or nm.startswith(cls+'-'):
            with zipfile.ZipFile(zf) as z:
                cn=next(n for n in z.namelist() if n.endswith('.csv') and n not in SKIP)
                with z.open(cn) as fh: sessions.append((fine(cls), prep(pd.read_csv(fh))))
            break

def build(apply_me):
    rows,y,g=[],[],[]; sid=0
    for cls,df in sessions:
        for w in wins(df):
            src=me_mask(w) if apply_me else w
            if len(src)>50: rows.append(feat(src)); y.append(cls); g.append(sid)
        sid+=1
    return pd.DataFrame(rows), np.array(y), np.array(g)

def svm(): return Pipeline([('sc',StandardScaler()),('svm',SVC(kernel='rbf',C=10,class_weight='balanced',random_state=42))])

def importance(X,y,g):
    cols=list(X.columns); imp=np.zeros(len(cols)); n=0
    for tr,te in GroupShuffleSplit(n_splits=5,test_size=0.3,random_state=42).split(X.values,y,g):
        if len(np.unique(y[tr]))<3: continue
        m=svm(); m.fit(X.iloc[tr],y[tr])
        r=permutation_importance(m,X.iloc[te],y[te],n_repeats=8,random_state=42,scoring='accuracy')
        imp+=r.importances_mean; n+=1
    return pd.Series(imp/max(n,1), index=cols).sort_values(ascending=False)

DATAW={}
for me in (False, True):
    X,y,g = build(me)
    imp = importance(X,y,g)
    sel = list(imp[imp>=0].index)
    DATAW[me] = dict(X=X, y=y, g=g, imp=imp, sel=sel)
    print(f"ME {'an ' if me else 'aus'}: {len(X)} Fenster · {len(np.unique(g))} Sessions · "
          f"SELECTED {len(sel)}/{X.shape[1]}")

---
## 3. Alle Konfigurationen auswerten

In [ ]:
def loso(X,y,g):
    yt,yp,mt,mp=[],[],[],[]
    for tr,te in LeaveOneGroupOut().split(X,y,g):
        if len(np.unique(y[tr]))<3: continue
        m=svm(); m.fit(X[tr],y[tr]); pr=m.predict(X[te])
        yt.extend(y[te]); yp.extend(pr)
        v,c=np.unique(pr,return_counts=True); mp.append(v[c.argmax()]); mt.append(y[te][0])
    return accuracy_score(yt,yp), accuracy_score(mt,mp), (np.array(yt),np.array(yp),np.array(mt),np.array(mp))
def split80(X,y):
    a=[]
    for s in range(5):
        Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=s)
        m=svm(); m.fit(Xtr,ytr); a.append(accuracy_score(yte,m.predict(Xte)))
    return float(np.mean(a))

results={}; preds={}
for me in (False,True):
    d=DATAW[me]; Xv=d['X'].values
    for setname, cols in [('PLUS', list(d['X'].columns)), ('SELECTED', d['sel'])]:
        Xs = d['X'][cols].values
        l, pm, pr = loso(Xs, d['y'], d['g'])
        s80 = split80(Xs, d['y'])
        key=(me,setname)
        results[key]=dict(loso=l, meal=pm, split=s80, nfeat=len(cols))
        preds[key]=pr
        print(f"ME {'an ' if me else 'aus'} {setname:8s} ({len(cols):2d}): "
              f"LOSO {l:.1%} · Mahlzeit {pm:.1%} · 80/20 {s80:.1%}")

best = max(results, key=lambda k: results[k]['loso'])
print(f"\nBestes LOSO: ME {'an' if best[0] else 'aus'} + {best[1]} "
      f"→ {results[best]['loso']:.1%} / Mahlzeit {results[best]['meal']:.1%}")

---
## 4. Grafik 1 — Vergleichsmatrix (warum SELECTED + LOSO)

In [ ]:
labels=['ME aus / PLUS-52','ME aus / SELECTED','ME an / PLUS-52','ME an / SELECTED']
keys=[(False,'PLUS'),(False,'SELECTED'),(True,'PLUS'),(True,'SELECTED')]
metrics=[('split','80/20-Split','#bab0ac'),('loso','LOSO (ehrlich)','#e15759'),('meal','Per-Mahlzeit','#59a14f')]
x=np.arange(len(keys)); w=0.26
fig,ax=plt.subplots(figsize=(11,5.8))
for i,(mkey,mlabel,col) in enumerate(metrics):
    vals=[results[k][mkey] for k in keys]
    bars=ax.bar(x+(i-1)*w, vals, w, label=mlabel, color=col, edgecolor='white')
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2, v+0.008, f'{v:.0%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
bi=keys.index(best)
ax.text(x[bi], 1.09, 'bestes Modell', ha='center', va='bottom', fontsize=11, fontweight='bold',
        color='#1a1a2e', bbox=dict(boxstyle='round,pad=0.3', fc='#f1c40f', ec='none'))
ax.text(0.5, 1.045, 'Feature Selection (PLUS -> SELECTED) hebt das ehrliche LOSO (rote Balken)',
        transform=ax.transAxes, ha='center', fontsize=9.5, color='#e15759', fontstyle='italic')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0,1.20); ax.set_ylabel('Accuracy (3 Speisen)')
ax.set_title('SVM — Feature-Set × Movement Exclusion × Bewertung', fontweight='bold', fontsize=13)
ax.legend(loc='lower right', framealpha=.95)
plt.tight_layout()
plt.savefig('../reports/images/final_modellvergleich.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Grafik 2 — Feature-Importance: *warum* SELECTED besser generalisiert

Group-aware Permutation Importance des besten Modellsatzes. Grün = gewählt, grau = verworfen,
⭐ = neues Kau-Dynamik-Feature. **Verworfen werden die absolute-Orientierungs-Features**
(session-spezifisches Rauschen) — gewählt wird die Kau-*Dynamik*.

In [ ]:
me_b = best[0]
imp = DATAW[me_b]['imp']; sel=set(DATAW[me_b]['sel'])
order = imp.index[::-1]
vals = imp[order].values
colors = ['#59a14f' if f in sel else '#d0d0d0' for f in order]
edges  = ['#b07aa1' if f in NEW else 'white' for f in order]
fig,ax=plt.subplots(figsize=(8.5,12))
ax.barh(range(len(order)), vals, color=colors, edgecolor=edges, linewidth=1.6)
ax.set_yticks(range(len(order))); ax.set_yticklabels(list(order), fontsize=7.5)
ax.axvline(0, color='black', lw=.8)
ax.set_xlabel('Permutation Importance (group-aware, Cross-Session)')
ax.set_title('grün = gewählt  ·  grau = verworfen  ·  lila Rand = neues Dynamik-Feature', fontsize=9.5, color='#555', pad=10)
fig.suptitle(f"Feature-Importance — bestes Modell (ME {'an' if me_b else 'aus'})", fontweight='bold', fontsize=13, y=0.997)
plt.tight_layout(rect=[0,0,1,0.975])
plt.savefig('../reports/images/final_featureimportance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Verworfen:', ', '.join(f for f in imp.index if f not in sel))

---
## 6. Grafik 3 — Bestes Modell: Confusion (3 Speisen) UND End-to-End inkl. Still

Zuerst die reine Speise-Klassifikation (Stufe 2). Danach die **volle Pipeline inkl. Still**
(Stufe 1 RF -> Stufe 2 SVM): das zeigt das Gesamtsystem mit der Still-Klasse.

In [ ]:
yt,yp,mt,mp = preds[best]
fig,axes=plt.subplots(1,2,figsize=(11,4.4))
sns.heatmap(confusion_matrix(yt,yp,labels=SPEC),annot=True,fmt='d',cmap='Blues',
            xticklabels=SPEC,yticklabels=SPEC,ax=axes[0],cbar=False,annot_kws={'size':12,'weight':'bold'})
axes[0].set_title(f'Stufe 2 pro Fenster - LOSO {results[best]["loso"]:.0%}',fontweight='bold')
axes[0].set_xlabel('Vorhergesagt'); axes[0].set_ylabel('Wahr')
sns.heatmap(confusion_matrix(mt,mp,labels=SPEC),annot=True,fmt='d',cmap='Greens',
            xticklabels=SPEC,yticklabels=SPEC,ax=axes[1],cbar=False,annot_kws={'size':12,'weight':'bold'})
axes[1].set_title(f'Stufe 2 pro Mahlzeit - Votum {results[best]["meal"]:.0%}',fontweight='bold')
axes[1].set_xlabel('Vorhergesagt'); axes[1].set_ylabel('Wahr')
me_b,set_b=best
plt.suptitle(f'Bestes Modell: SVM - ME {"an" if me_b else "aus"} - {set_b} ({results[best]["nfeat"]} Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/images/final_best_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

# --- End-to-End inkl. Still: volle 2-Stufen-Pipeline (bestes Modell, ME an) ---
from sklearn.ensemble import RandomForestClassifier
ALLRAW=['Apfel','Kaugummi','Skyr','Skyr_Zimt','Still','Essen']
TO_C={'Apfel':'Essen','Kaugummi':'Essen','Skyr':'Essen','Still':'Still','Essen':'Essen'}
S1F=['stillness_ratio','magnitude_max','lin_y_mean','lin_y_std','yaw_mean']
LAB=['Still','Apfel','Kaugummi','Skyr']
rows=[]; ya=[]; ga=[]; sid=0
for zf in sorted(DATA.glob('*.zip')):
    nm=zf.name
    for cls in sorted(ALLRAW,key=len,reverse=True):
        if nm.startswith(cls+'_') or nm.startswith(cls+'-'):
            with zipfile.ZipFile(zf) as z:
                cn=next(n for n in z.namelist() if n.endswith('.csv') and n not in SKIP)
                with z.open(cn) as fh: dd=prep(pd.read_csv(fh))
            for w in wins(dd):
                src=me_mask(w)
                if len(src)>50: rows.append(feat(src)); ya.append(fine(cls)); ga.append(sid)
            sid+=1; break
Xa=pd.DataFrame(rows); ya=np.array(ya); ga=np.array(ga); yca=np.array([TO_C[c] for c in ya])
fs2=DATAW[True]['sel']
et,ep,mt2,mp2=[],[],[],[]
for tr,te in LeaveOneGroupOut().split(Xa.values,ya,ga):
    if ya[te][0] not in LAB: continue
    m1=RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=42); m1.fit(Xa[S1F].values[tr],yca[tr])
    pc=m1.predict(Xa[S1F].values[te])
    sm=np.isin(ya[tr],SPEC)
    m2=Pipeline([('sc',StandardScaler()),('svm',SVC(kernel='rbf',C=10,class_weight='balanced',random_state=42))]); m2.fit(Xa[fs2].values[tr][sm],ya[tr][sm])
    pr=[m2.predict(Xa[fs2].values[[i]])[0] if c=='Essen' else 'Still' for i,c in zip(te,pc)]
    et.extend(ya[te]); ep.extend(pr)
    v,cn=np.unique(pr,return_counts=True); mp2.append(v[cn.argmax()]); mt2.append(ya[te][0])
fig,axes=plt.subplots(1,2,figsize=(12,4.6))
sns.heatmap(confusion_matrix(et,ep,labels=LAB),annot=True,fmt='d',cmap='Blues',xticklabels=LAB,yticklabels=LAB,ax=axes[0],cbar=False,annot_kws={'size':11,'weight':'bold'})
axes[0].set_title(f'End-to-End pro Fenster ({accuracy_score(et,ep):.0%})',fontweight='bold'); axes[0].set_xlabel('Vorhergesagt'); axes[0].set_ylabel('Wahr')
sns.heatmap(confusion_matrix(mt2,mp2,labels=LAB),annot=True,fmt='d',cmap='Greens',xticklabels=LAB,yticklabels=LAB,ax=axes[1],cbar=False,annot_kws={'size':11,'weight':'bold'})
axes[1].set_title(f'End-to-End pro Mahlzeit ({accuracy_score(mt2,mp2):.0%})',fontweight='bold'); axes[1].set_xlabel('Vorhergesagt'); axes[1].set_ylabel('Wahr')
plt.suptitle('Volle Pipeline inkl. Still - Stufe 1 (RF) + Stufe 2 (SVM, selektiert), LOSO',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/images/final_endtoend_confusion.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'End-to-End: {accuracy_score(et,ep):.1%} pro Fenster, {accuracy_score(mt2,mp2):.1%} pro Mahlzeit')

---
## 7. Fazit — warum dieses Modell

- **Feature Selection ist der entscheidende Hebel:** Sie hebt das *ehrliche* LOSO deutlich
  (PLUS → SELECTED), indem sie session-spezifische Rausch-Features (absolute Orientierung
  wie `roll_mean`, `yaw_mean`) entfernt. Mehr Features ≠ besser.
- **Was bleibt, ist Kau-Dynamik** (Jerk, Kaufrequenz, Rhythmus, Knusper-Energie) — generalisiert
  über Sessions, weil es *wie gekaut wird* beschreibt, nicht *wo das Handy liegt*.
- **LOSO ist der ehrliche Maßstab** (80/20 überschätzt durch Leakage); per Mahlzeit erreicht das
  beste Modell den Real-Use-Wert.

> **Bestes Modell:** SVM + Kau-Dynamik-Features + group-aware Feature Selection — sauber per
> LOSO gemessen, stark per Mahlzeit. *(Hinweis: die Selektion nutzt dieselben Sessions wie das
> berichtete LOSO → leicht optimistisch; der Trend ist robust.)*